# Notebook 01 — Pratique SQL avec psycopg2
**Base :** `cyclistic_db` | **Table :** `cyclistic_clean`  
**Objectif :** Exécuter les 12 requêtes d'analyse Cyclistic depuis Python et lire les résultats en DataFrame pandas.

---
## Structure du notebook
1. Connexion à PostgreSQL
2. Fonction utilitaire `run_query()`
3. Requête 01 — Vue d'ensemble
4. Requête 02 — Durée moyenne par jour de la semaine
5. Requête 03 — Trajets par mois
6. Requête 04 — Trajets par saison
7. Requête 05 — Types de vélos
8. Requête 06 — Top 10 stations membres
9. Requête 07 — Top 10 stations casual
10. Requête 08 — Top 10 stations d'arrivée
11. Requête 09 — Répartition par heure
12. Requête 10 — Jour le plus populaire (mode)
13. Requête 11 — Résumé pivot membres vs casual
14. Requête 12 — Stations partagées (JOIN)
15. Exercices pratiques


---
## 1. Connexion à PostgreSQL

`psycopg2.connect()` ouvre la ligne de communication entre Python et PostgreSQL.  
On stocke la connexion dans `conn` et on crée un `cursor` pour envoyer des requêtes.

In [ ]:
import psycopg2
import pandas as pd

# Paramètres de connexion
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="cyclistic_db",
    user="postgres",
    password="postgres"
)

print("Connexion réussie !")
print(f"Serveur PostgreSQL : {conn.server_version}")

---
## 2. Fonction utilitaire `run_query()`

Plutôt que de répéter `cursor → execute → fetchall` à chaque fois,  
on crée une fonction réutilisable qui retourne directement un **DataFrame pandas**.

> **Pourquoi pandas ?** Un DataFrame est un tableau avec des noms de colonnes,  
> des méthodes de filtrage, de tri, d'affichage — beaucoup plus pratique qu'une liste de tuples bruts.

In [ ]:
def run_query(sql, conn=conn):
    """
    Exécute une requête SQL et retourne un DataFrame pandas.
    
    Paramètres
    ----------
    sql  : str  — la requête SQL à exécuter
    conn : connexion psycopg2 active
    
    Retourne
    --------
    pd.DataFrame avec les résultats
    """
    cur = conn.cursor()          # Crée le curseur (le "stylo")
    cur.execute(sql)             # Envoie la requête au serveur
    rows = cur.fetchall()        # Récupère toutes les lignes de résultats
    cols = [d[0] for d in cur.description]  # Récupère les noms de colonnes
    cur.close()                  # Ferme le curseur
    return pd.DataFrame(rows, columns=cols)  # Construit le DataFrame

print("Fonction run_query() prête.")

---
## Requête 01 — Vue d'ensemble : statistiques descriptives

**Question :** Combien de trajets ? Quelle durée moyenne / médiane / max par type d'utilisateur ?

**Fonctions SQL utilisées :**
- `COUNT(*)` → nombre total de lignes
- `AVG()` → moyenne
- `PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ...)` → médiane (50e percentile)
- `MAX()`, `MIN()`, `STDDEV()` → autres statistiques
- `ROUND()` → arrondir à 2 décimales
- `GROUP BY` → regrouper par type d'utilisateur

In [ ]:
q01 = """
SELECT
    member_casual,
    COUNT(*)                                                         AS total_rides,
    ROUND(AVG(ride_length), 2)                                       AS avg_ride_length_min,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP
          (ORDER BY ride_length)::numeric, 2)                        AS median_ride_length_min,
    ROUND(MAX(ride_length)::numeric, 2)                              AS max_ride_length_min,
    ROUND(MIN(ride_length)::numeric, 2)                              AS min_ride_length_min,
    ROUND(STDDEV(ride_length)::numeric, 2)                           AS std_ride_length_min
FROM cyclistic_clean
GROUP BY member_casual
ORDER BY member_casual;
"""

df01 = run_query(q01)
df01

**Lecture des résultats :**
- Les casual riders ont une durée moyenne ~2,5× supérieure aux membres
- La médiane est plus faible que la moyenne → distribution asymétrique (quelques très longs trajets tirent la moyenne vers le haut)
- `STDDEV` élevé chez les casual → grande variabilité dans leurs trajets

---
## Requête 02 — Durée et volume par jour de la semaine

**Question :** Comment varient les trajets et leur durée selon le jour de la semaine ?  
**Nouveau :** `CASE WHEN` pour transformer le numéro de jour en nom lisible.

In [ ]:
q02 = """
SELECT
    member_casual,
    day_of_week,
    CASE day_of_week
        WHEN 1 THEN 'Dimanche'
        WHEN 2 THEN 'Lundi'
        WHEN 3 THEN 'Mardi'
        WHEN 4 THEN 'Mercredi'
        WHEN 5 THEN 'Jeudi'
        WHEN 6 THEN 'Vendredi'
        WHEN 7 THEN 'Samedi'
    END                                      AS day_name,
    COUNT(*)                                 AS total_rides,
    ROUND(AVG(ride_length)::numeric, 2)      AS avg_ride_length_min
FROM cyclistic_clean
GROUP BY member_casual, day_of_week
ORDER BY member_casual, day_of_week;
"""

df02 = run_query(q02)
df02

In [ ]:
# Pivoter pour comparer membres vs casual côte à côte
pivot_rides = df02.pivot(index='day_name', columns='member_casual', values='total_rides')
print("Nombre de trajets par jour :")
print(pivot_rides.to_string())

---
## Requête 03 — Trajets par mois

**Question :** Quelle est l'évolution mensuelle des trajets sur 2019–2020 ?  
**Nouveau :** `GROUP BY` sur deux colonnes (`year`, `month`) pour une granularité mensuelle.

In [ ]:
q03 = """
SELECT
    member_casual,
    year,
    month,
    COUNT(*)                                 AS total_rides,
    ROUND(AVG(ride_length)::numeric, 2)      AS avg_ride_length_min
FROM cyclistic_clean
GROUP BY member_casual, year, month
ORDER BY member_casual, year, month;
"""

df03 = run_query(q03)
df03.head(12)

---
## Requête 04 — Trajets par saison

**Question :** Quelle saison concentre le plus de trajets ? Quelle est la part (%) de chaque saison ?  
**Nouveau :** `OVER (PARTITION BY ...)` — fonction de fenêtrage pour calculer un % sans sous-requête.

In [ ]:
q04 = """
SELECT
    member_casual,
    season,
    COUNT(*)                                                              AS total_rides,
    ROUND(AVG(ride_length)::numeric, 2)                                   AS avg_ride_length_min,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY member_casual)
    , 1)                                                                  AS pct_of_user_type
FROM cyclistic_clean
GROUP BY member_casual, season
ORDER BY member_casual,
    CASE season
        WHEN 'Spring' THEN 1
        WHEN 'Summer' THEN 2
        WHEN 'Fall'   THEN 3
        WHEN 'Winter' THEN 4
    END;
"""

df04 = run_query(q04)
df04

---
## Requête 05 — Types de vélos utilisés

**Question :** Quelle proportion de chaque type de vélo pour membres vs casual ?

In [ ]:
q05 = """
SELECT
    member_casual,
    rideable_type,
    COUNT(*)                                                              AS total_rides,
    ROUND(AVG(ride_length)::numeric, 2)                                   AS avg_ride_length_min,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY member_casual)
    , 1)                                                                  AS pct_of_user_type
FROM cyclistic_clean
GROUP BY member_casual, rideable_type
ORDER BY member_casual, total_rides DESC;
"""

df05 = run_query(q05)
df05

---
## Requêtes 06 & 07 — Top 10 stations de départ

**Question :** Quelles sont les stations les plus fréquentées par chaque type ?  
**Nouveau :** `LIMIT` pour ne garder que les N premières lignes après tri.

In [ ]:
q06 = """
SELECT
    start_station_name,
    COUNT(*) AS total_rides
FROM cyclistic_clean
WHERE member_casual = 'member'
  AND start_station_name IS NOT NULL
GROUP BY start_station_name
ORDER BY total_rides DESC
LIMIT 10;
"""

df06 = run_query(q06)
print("Top 10 stations — MEMBRES :")
df06

In [ ]:
q07 = """
SELECT
    start_station_name,
    COUNT(*) AS total_rides
FROM cyclistic_clean
WHERE member_casual = 'casual'
  AND start_station_name IS NOT NULL
GROUP BY start_station_name
ORDER BY total_rides DESC
LIMIT 10;
"""

df07 = run_query(q07)
print("Top 10 stations — CASUAL :")
df07

---
## Requête 08 — Top 10 stations d'arrivée

**Question :** Où les utilisateurs terminent-ils leurs trajets ?

In [ ]:
q08 = """
SELECT
    member_casual,
    end_station_name,
    COUNT(*) AS total_rides
FROM cyclistic_clean
WHERE end_station_name IS NOT NULL
GROUP BY member_casual, end_station_name
ORDER BY member_casual, total_rides DESC
LIMIT 20;
"""

df08 = run_query(q08)
df08

---
## Requête 09 — Répartition par heure de départ

**Question :** À quelle heure les utilisateurs démarrent-ils leurs trajets ?  
C'est ici qu'on voit clairement le **double pic 8h/17h** des membres (rush hours).

In [ ]:
q09 = """
SELECT
    member_casual,
    hour_of_day,
    COUNT(*)                                 AS total_rides,
    ROUND(AVG(ride_length)::numeric, 2)      AS avg_ride_length_min
FROM cyclistic_clean
GROUP BY member_casual, hour_of_day
ORDER BY member_casual, hour_of_day;
"""

df09 = run_query(q09)

# Afficher les heures de pointe uniquement
print("Heures de pointe membres (entre 7h et 19h) :")
mask = (df09['member_casual'] == 'member') & (df09['hour_of_day'].between(7, 19))
print(df09[mask].to_string(index=False))

---
## Requête 10 — Jour le plus populaire (mode)

**Question :** Quel est le jour de la semaine le plus populaire pour chaque type ?  
**Nouveau :** `CTE (WITH ... AS)` + `ROW_NUMBER() OVER` — requête avancée en deux étapes.

In [ ]:
q10 = """
-- Étape 1 : CTE qui calcule le rang de chaque jour par nombre de trajets
WITH ranked_days AS (
    SELECT
        member_casual,
        day_of_week,
        COUNT(*)                    AS rides,
        ROW_NUMBER() OVER (
            PARTITION BY member_casual
            ORDER BY COUNT(*) DESC
        )                           AS rnk
    FROM cyclistic_clean
    GROUP BY member_casual, day_of_week
)
-- Étape 2 : ne garder que le rang 1 (le jour le plus populaire)
SELECT
    member_casual,
    day_of_week,
    CASE day_of_week
        WHEN 1 THEN 'Dimanche'
        WHEN 2 THEN 'Lundi'
        WHEN 3 THEN 'Mardi'
        WHEN 4 THEN 'Mercredi'
        WHEN 5 THEN 'Jeudi'
        WHEN 6 THEN 'Vendredi'
        WHEN 7 THEN 'Samedi'
    END                             AS most_popular_day,
    rides
FROM ranked_days
WHERE rnk = 1
ORDER BY member_casual;
"""

df10 = run_query(q10)
df10

---
## Requête 11 — Résumé pivot membres vs casual

**Question :** Vue condensée en une seule ligne pour comparer les deux groupes.  
**Nouveau :** `SUM(CASE WHEN ...)` — technique de pivot horizontal en SQL pur.

In [ ]:
q11 = """
SELECT
    SUM(CASE WHEN member_casual = 'member' THEN 1 ELSE 0 END)   AS member_rides,
    SUM(CASE WHEN member_casual = 'casual' THEN 1 ELSE 0 END)   AS casual_rides,
    ROUND(
        AVG(CASE WHEN member_casual = 'member' THEN ride_length END)::numeric
    , 2)                                                         AS member_avg_min,
    ROUND(
        AVG(CASE WHEN member_casual = 'casual' THEN ride_length END)::numeric
    , 2)                                                         AS casual_avg_min,
    ROUND(
        SUM(CASE WHEN member_casual = 'member' THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
    , 1)                                                         AS member_pct,
    ROUND(
        SUM(CASE WHEN member_casual = 'casual' THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
    , 1)                                                         AS casual_pct
FROM cyclistic_clean;
"""

df11 = run_query(q11)
df11

---
## Requête 12 — Stations partagées (JOIN)

**Question :** Quelles stations sont populaires pour LES DEUX types d'utilisateurs ?  
**Nouveau :** `JOIN` entre deux CTEs — technique avancée pour croiser deux ensembles de données.

In [ ]:
q12 = """
-- CTE 1 : top stations membres
WITH member_stations AS (
    SELECT start_station_name, COUNT(*) AS member_rides
    FROM cyclistic_clean
    WHERE member_casual = 'member' AND start_station_name IS NOT NULL
    GROUP BY start_station_name
),
-- CTE 2 : top stations casual
casual_stations AS (
    SELECT start_station_name, COUNT(*) AS casual_rides
    FROM cyclistic_clean
    WHERE member_casual = 'casual' AND start_station_name IS NOT NULL
    GROUP BY start_station_name
)
-- JOIN : ne garder que les stations présentes dans les deux groupes
SELECT
    m.start_station_name,
    m.member_rides,
    c.casual_rides,
    m.member_rides + c.casual_rides   AS total_rides
FROM member_stations m
JOIN casual_stations c USING (start_station_name)
ORDER BY total_rides DESC
LIMIT 20;
"""

df12 = run_query(q12)
df12

---
## Exercices pratiques

Essaie de résoudre ces requêtes par toi-même avant de regarder les solutions !

### Exercice 1
Trouve le **mois avec le plus grand écart de durée moyenne** entre membres et casual.

In [ ]:
# Ton code ici
ex1 = """
-- Écris ta requête ici

"""
# run_query(ex1)

### Exercice 2
Calcule le **nombre de trajets le week-end vs en semaine** pour chaque type d'utilisateur,  
et exprime chaque part en pourcentage.

*Indice : `day_of_week IN (1, 7)` = week-end dans notre encodage (1=Dimanche, 7=Samedi)*

In [ ]:
# Ton code ici
ex2 = """
-- Écris ta requête ici

"""
# run_query(ex2)

### Exercice 3 (avancé)
Pour chaque station du **top 10 casual**, calcule :  
- le nombre de trajets casual  
- le nombre de trajets membres sur la même station  
- le ratio casual/membre

*Indice : utilise un JOIN comme dans la requête 12*

In [ ]:
# Ton code ici
ex3 = """
-- Écris ta requête ici

"""
# run_query(ex3)

---
## Fermeture de la connexion

Toujours fermer la connexion à la fin du notebook — comme raccrocher le téléphone.

In [ ]:
conn.close()
print("Connexion fermée.")